In [1]:
from graph_transformer_long_range_niches.tl.wandb import load_and_log
from graph_transformer_long_range_niches.tl.load_config import Config
from graph_transformer_long_range_niches._paths import CFG_FILES

from pathlib import Path
import wandb
import torch
import os

# Log and load data 

In [2]:
cfg_path = Path(CFG_FILES, 'He23/he23_gnntrans_niche.yaml')
cfg = Config(cfg_path)

Load cfg file...
{'out_dir': 'results', 'wandb': {'use': True, 'project_name': 'GTLongRange', 'run_idx': None}, 'model': {'model_type': 'gnn-transformer', 'n_epochs': 100}, 'optim': {'lr': 0.001, 'wd': '1e-3', 'warm_up': 10, 'seed': 42}, 'dataset': {'h5ad_data': '/lustre/groups/ml01/projects/2024_spatial_long_range_GT_francesca.drummer/unprocessed_data/he22_cosmx_human_lung.h5ad', 'name': 'he23_niche_window', 'description': "CosmX Lung data from He23 with graphs for each .obs['window'] and prediction node .obs['niche'].", 'prediction_task': 'node', 'prediction_obs': 'niche', 'library_key': 'window', 'subset_dict': {}, 'spatial_neigbors_kwargs': {'radius': 30, 'coord_type': 'generic'}, 'batch_size': 20, 'train_size': 0.8, 'val_size': 0.2, 'test_size': 0.0}, 'gnn': {'gnn_type': 'GCN', 'num_layers': 2, 'hidden_dim': 256, 'embed_dim': 256, 'dropout': 0.1}, 'transformer': {'d_model': 128, 'n_heads': 4, 'dim_feedforward': 512, 'dropout': 0.3, 'num_layers': 4, 'activation_func': 'relu', 'num_

In [3]:
load_and_log(cfg)

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: francesca-drummer. Use `wandb login --relogin` to force relogin


Load PyG data...
No subset creation.
[Data(x=[1169, 960], edge_index=[2, 42], y=[1169, 9]), Data(x=[2573, 960], edge_index=[2, 244], y=[2573, 9]), Data(x=[2527, 960], edge_index=[2, 206], y=[2527, 9])]
375
train ds: 300, val ds: 75


/home/icb/francesca.drummer/tools/apps/mamba/envs/exphormer_mamba/lib/python3.11/site-packages/torch_geometric/data/lightning/datamodule.py:43: The 'shuffle=True' option is ignored in 'LightningDataset'. Remove it from the argument list to disable this warning


Load data from WandB

In [6]:
with wandb.init(project=cfg.get('wandb/project_name'), job_type="load-data", name = 'data_'+cfg.get('dataset/name')) as run:
    name = cfg.get('dataset/name')
    data_artifact = run.use_artifact(f'{name}:latest')

Thread HandlerThread:
Traceback (most recent call last):
  File "/home/icb/francesca.drummer/tools/apps/mamba/envs/exphormer_mamba/lib/python3.11/site-packages/wandb/sdk/internal/internal_util.py", line 49, in run
    self._run()
  File "/home/icb/francesca.drummer/tools/apps/mamba/envs/exphormer_mamba/lib/python3.11/site-packages/wandb/sdk/internal/internal_util.py", line 100, in _run
    self._process(record)
  File "/home/icb/francesca.drummer/tools/apps/mamba/envs/exphormer_mamba/lib/python3.11/site-packages/wandb/sdk/internal/internal.py", line 279, in _process
    self._hm.handle(record)
  File "/home/icb/francesca.drummer/tools/apps/mamba/envs/exphormer_mamba/lib/python3.11/site-packages/wandb/sdk/internal/handler.py", line 138, in handle
    handler(record)
  File "/home/icb/francesca.drummer/tools/apps/mamba/envs/exphormer_mamba/lib/python3.11/site-packages/wandb/sdk/internal/handler.py", line 148, in handle_request
    handler(record)
  File "/home/icb/francesca.drummer/tools

Problem at: /home/icb/francesca.drummer/tools/apps/mamba/envs/exphormer_mamba/lib/python3.11/site-packages/wandb/sdk/wandb_init.py 849 getcaller



KeyboardInterrupt



In [13]:
data = data_artifact.download()

wandb: Downloading large artifact he23_niche_window:latest, 2852.93MB. 2 files... 
wandb:   2 of 2 files downloaded.  
Done. 0:1:12.2


In [13]:
def read(data_dir, split):
    filename = split + ".pt"
    pyg_data = torch.load(os.path.join(data_dir, filename))
    return pyg_data # list with PyG Data objects

In [ ]:
data_dir = '/home/icb/francesca.drummer/1-Projects/GT-long-range-niches/docs/notebooks/artifacts/he23_niche_window:v0'

In [14]:
pyg_data_val = read(data_dir, 'validation')

# Download model

In [3]:
name = cfg.get('dataset/name') + '_' + cfg.get('model/model_type')
name

'he23_niche_window_gnn-transformer'

In [8]:
# name = cfg.get('dataset/name') + cfg.get('model/model_type')
# with wandb.init(project=cfg.get('wandb/project_name'), 
#                 job_type="load_model", 
#                 #name = cfg.get('dataset/name') + cfg.get('model/model_type')) as run:
#                 name = 'load_model_' + name) as run:
#     artifact = run.use_artifact('model-srw7c0r8:best', type='model')
#     artifact_dir = artifact.download()
#     model_path = f'{artifact_dir}/model.pth'
#     model = torch.load(model_path)

In [5]:
import wandb
run = wandb.init()
artifact = run.use_artifact('francesca-drummer/GTLongRange/model-srw7c0r8:v99', type='model')
artifact_dir = artifact.download()

wandb:   1 of 1 files downloaded.  


In [7]:
model_path = f'{artifact_dir}/model.ckpt'
model = torch.load(model_path)

# Run evaluation

How does Ale retrieve the attention map? [https://github.com/theislab/spatial-transformer/blob/c882e5a1a43b07a17ba8ea32fc59344cd01047cf/src/spatra/model/_transformer.py#L363](https://github.com/theislab/spatial-transformer/blob/c882e5a1a43b07a17ba8ea32fc59344cd01047cf/src/spatra/model/_transformer.py#L363)

Extract self-attention maps from nn.TransformerEncoder: [https://discuss.pytorch.org/t/extracting-self-attention-maps-from-nn-transformerencoder/139998](https://discuss.pytorch.org/t/extracting-self-attention-maps-from-nn-transformerencoder/139998)